In [7]:
import os
import json
import shutil
import random

import matplotlib.pyplot as plt

# ==================================================
# PATHS
# ==================================================

SCRIPT_DIR = os.getcwd()

BASE_DIR = os.path.abspath(
    os.path.join(
        SCRIPT_DIR,
        "..",
        "search-dataset",
        "visual_search_dataset_counterfactual",
    )
)

# ==================================================
# IMAGE SETTINGS
# ==================================================

IMAGE_SIZE = 448
DPI = 112
FIGSIZE = (
    IMAGE_SIZE / DPI,
    IMAGE_SIZE / DPI,
)

FONT_SIZE = 24

DISTRACTOR_COUNTS = [
    0, 5, 10, 15, 20,
    25, 30, 35, 40, 45, 50
]

CONDITIONS = [
    "disjunctive",
    "conjunctive",
]

IMAGES_PER_BIN = 100
NUM_TARGET_PRESENT = IMAGES_PER_BIN // 2

SEED = 42
random.seed(SEED)

# ==================================================
# CANONICAL TARGET POSITIONS
# ==================================================

def generate_canonical_target_positions():

    positions = []

    while len(positions) < NUM_TARGET_PRESENT:

        pos = (
            random.uniform(1.0, 9.0),
            random.uniform(1.0, 9.0),
        )

        if all(
            (pos[0] - p[0]) ** 2
            + (pos[1] - p[1]) ** 2
            > 0.65
            for p in positions
        ):
            positions.append(pos)

    return positions


# ==================================================
# SINGLE TRIAL
# ==================================================

def generate_search_trial(
    condition,
    target_present,
    num_distractors,
    save_path,
    forced_target_pos=None,
):

    fig, ax = plt.subplots(
        figsize=FIGSIZE,
        dpi=DPI,
    )

    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=1,
    )

    ax.set_xlim(0.5, 9.5)
    ax.set_ylim(0.5, 9.5)
    ax.set_aspect("equal")
    ax.axis("off")

    occupied = []

    metadata = {
        "condition": condition,
        "target_present": target_present,
        "num_distractors": num_distractors,
        "image_size": IMAGE_SIZE,
        "dpi": DPI,
        "font_size": FONT_SIZE,
        "objects": [],
    }

    # ==================================================
    # POSITION SAMPLING
    # ==================================================

    def get_random_position():

        for _ in range(1000):

            pos = (
                random.uniform(1.0, 9.0),
                random.uniform(1.0, 9.0),
            )

            threshold = (
                1.0
                if num_distractors <= 25
                else 0.8
            )

            if all(
                (
                    (pos[0] - p[0]) ** 2
                    + (pos[1] - p[1]) ** 2
                ) > threshold
                for p in occupied
            ):
                return pos

        return (
            random.uniform(1.0, 9.0),
            random.uniform(1.0, 9.0),
        )

    # ==================================================
    # DRAW LETTER + STORE METADATA
    # ==================================================

    def add_letter(
        pos,
        shape,
        color,
        role,
    ):

        txt = ax.text(
            pos[0],
            pos[1],
            shape,
            color=color,
            fontsize=FONT_SIZE,
            ha="center",
            va="center",
            fontweight="bold",
            fontfamily="DejaVu Sans Mono",
        )

        fig.canvas.draw()

        renderer = fig.canvas.get_renderer()

        bbox = txt.get_window_extent(
            renderer
        )

        metadata["objects"].append({
            "role": role,
            "shape": shape,
            "color": color,

            "position_plot": [
                float(pos[0]),
                float(pos[1]),
            ],

            "position_pixel": [
                float(
                    bbox.x0 + bbox.width / 2
                ),
                float(
                    bbox.y0 + bbox.height / 2
                ),
            ],

            "bbox_pixel": {
                "x0": float(bbox.x0),
                "y0": float(bbox.y0),
                "x1": float(bbox.x1),
                "y1": float(bbox.y1),
                "width": float(bbox.width),
                "height": float(bbox.height),
            },
        })

    # ==================================================
    # TARGET
    # ==================================================

    if target_present:

        if forced_target_pos is None:
            pos = get_random_position()
        else:
            pos = forced_target_pos

        occupied.append(pos)

        shape = "L"
        color = "green"

        add_letter(
            pos,
            shape,
            color,
            role="target",
        )

    # ==================================================
    # DISTRACTORS
    # ==================================================

    for _ in range(num_distractors):

        pos = get_random_position()
        occupied.append(pos)

        if condition == "disjunctive":

            shape = "L"
            color = "red"

        else:

            shape, color = random.choice([
                ("L", "red"),
                ("T", "green"),
            ])

        add_letter(
            pos,
            shape,
            color,
            role="distractor",
        )

    # ==================================================
    # SAVE
    # ==================================================

    fig.savefig(
        save_path,
        dpi=DPI,
    )

    plt.close(fig)

    json_path = save_path.replace(
        ".png",
        ".json",
    )

    with open(json_path, "w") as f:
        json.dump(
            metadata,
            f,
            indent=2,
        )


# ==================================================
# GENERATE ENTIRE DATASET
# ==================================================

def generate_dataset():

    if os.path.exists(BASE_DIR):
        print("Deleting old dataset...")
        shutil.rmtree(BASE_DIR)

    os.makedirs(BASE_DIR)

    canonical_positions = (
        generate_canonical_target_positions()
    )

    for condition in CONDITIONS:

        print(f"\nGenerating {condition}")

        folder = os.path.join(
            BASE_DIR,
            condition,
        )

        os.makedirs(folder)

        for nd in DISTRACTOR_COUNTS:

            trial = 0

            # target present
            for i in range(
                NUM_TARGET_PRESENT
            ):

                fname = (
                    f"{condition}"
                    f"_distractors_{nd}"
                    f"_trial_{trial:03d}"
                    f"_target_true.png"
                )

                generate_search_trial(
                    condition=condition,
                    target_present=True,
                    num_distractors=nd,
                    save_path=os.path.join(
                        folder,
                        fname,
                    ),
                    forced_target_pos=(
                        canonical_positions[i]
                    ),
                )

                trial += 1

            # target absent
            for i in range(
                NUM_TARGET_PRESENT
            ):

                fname = (
                    f"{condition}"
                    f"_distractors_{nd}"
                    f"_trial_{trial:03d}"
                    f"_target_false.png"
                )

                generate_search_trial(
                    condition=condition,
                    target_present=False,
                    num_distractors=nd,
                    save_path=os.path.join(
                        folder,
                        fname,
                    ),
                )

                trial += 1

            print(
                f"  distractors={nd} done"
            )

        print(
            f"Finished {condition}"
        )

    print("\nDone.")


# ==================================================
# MAIN
# ==================================================

if __name__ == "__main__":
    generate_dataset()

Deleting old dataset...

Generating disjunctive
  distractors=0 done
  distractors=5 done
  distractors=10 done
  distractors=15 done
  distractors=20 done
  distractors=25 done
  distractors=30 done
  distractors=35 done
  distractors=40 done
  distractors=45 done
  distractors=50 done
Finished disjunctive

Generating conjunctive
  distractors=0 done
  distractors=5 done
  distractors=10 done
  distractors=15 done
  distractors=20 done
  distractors=25 done
  distractors=30 done
  distractors=35 done
  distractors=40 done
  distractors=45 done
  distractors=50 done
Finished conjunctive

Done.


In [8]:
from PIL import Image
from collections import Counter
import os

sizes = []

for root, _, files in os.walk(BASE_DIR):
    for f in files:
        if f.endswith(".png"):
            sizes.append(
                Image.open(
                    os.path.join(root, f)
                ).size
            )

print(Counter(sizes))

Counter({(448, 448): 2200})
